# pi-metaboqc correction benchmark against MetaNorm

This notebook evaluates the filtered feature matrix with every available pi-metaboqc correction candidate in Full and OOF modes, including the numba-backed standard and robust QC-RLSC implementations and QC-RFSC, then compares MetaNorm rLOESS, rGAM, and tGAM through rpy2. MetaNorm's default `keepScale=TRUE` is additive on the scale supplied to `metanormWorker`; because the package examples and model are intended for log-intensity data, the bridge uses `log2(raw)` as input and explicitly applies `exp2` to the returned log2 matrix before raw-scale QC-RSD calculations. MetaNorm retains all samples for prediction and correction, but excludes Blank samples from Full-mode fitting and every `QCcheck` comparison/refit. If rLOESS cannot extrapolate a Blank outside its batch fitting range, that Blank is retained at its original value and the count is recorded. The notebook also reports QC-RSD on the returned log2 scale so this nonlinear scale conversion is auditable. A separate `QConly=True` analysis fits drift using only QCs; this and Full mode are different estimands and are not expected to rank identically. It requires the `metaboqc` environment plus the R packages `metanorm` and `mgcv`. Each corrected matrix, parameter table, scale audit, QC-RSD table, paired feature comparison, structure metrics, and diagnostic figure is written below the input folder.

In [ ]:
from pathlib import Path
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "pimqc").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
INPUT = ROOT / "examples" / "tutorial_output" / "02_MV_Filtered" / "Filtered_Data_High-MV_Features.csv"
OUTPUT = INPUT.parent / "MetaNorm_Correction_Comparison"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT)
print("Output:", OUTPUT)

In [ ]:
# The filtered export stores five metadata rows followed by a sentinel row.
raw_csv = pd.read_csv(INPUT, header=None, dtype=object)
sentinel = np.flatnonzero(raw_csv.iloc[:, 0].astype(str).eq("Metabolite"))
if len(sentinel) != 1:
    raise ValueError("Could not locate the Metabolite sentinel row.")
sentinel = int(sentinel[0])
metadata_names = raw_csv.iloc[:sentinel, 0].astype(str).tolist()
sample_names = raw_csv.iloc[sentinel - 1, 1:].astype(str).to_numpy()
meta_df = pd.DataFrame(
    {name: raw_csv.iloc[idx, 1:].to_numpy() for idx, name in enumerate(metadata_names)}
)
meta_df["Inject Order"] = pd.to_numeric(meta_df["Inject Order"], errors="coerce")
intensity_df = raw_csv.iloc[sentinel + 1:, 1:].copy()
intensity_df.index = raw_csv.iloc[sentinel + 1:, 0].astype(str).to_numpy()
intensity_df.columns = sample_names
intensity_df = intensity_df.apply(pd.to_numeric, errors="coerce")
intensity_df = intensity_df.loc[~intensity_df.index.duplicated(keep="first")]

from pimqc.dataset.builder import build_dataset
from pimqc.processing.correction import MetaboIntCorrector

PIPELINE_PARAMS = {
    "MetaboInt": {
        "mode": "ESI+", "sample_name": "Sample Name",
        "sample_type": "Sample Type", "bio_group": "Bio Group",
        "batch": "Batch", "inject_order": "Inject Order",
        "sample_dict": {"Actual sample": "Sample", "Blank sample": "Blank", "QC sample": "QC"},
        "global_seed": 123,
    },
    "MetaboIntCorrector": {
        "loess_span": 0.5, "loess_degree": 1,
        "rlsc_span_selection": "fixed", "rlsc_span_grid": [0.3, 0.5, 0.7],
        "rlsc_min_qc": 7, "rlsc_robust_iterations": 3,
        # Explicit comparison settings keep the notebook reproducible and tractable.
        "rf_n_tree": 50, "serrf_n_tree": 20, "serrf_corr_features": 5,
        "cv_folds": 3, "ruv_k": 3, "waveica_components": 10,
        "waveica_cutoff": 0.1, "waveica_spline_knots": 5,
        "global_seed": 123, "n_jobs": 1, "regression_backend": "threading",
    },
}
data_obj = build_dataset(meta_info=meta_df, int_df=intensity_df, pipeline_params=PIPELINE_PARAMS)
corr_obj = MetaboIntCorrector(data_obj, pipeline_params=PIPELINE_PARAMS)
corr_obj.attrs.update(data_obj.attrs)
batch_col = corr_obj.attrs.get("batch", "Batch")
sample_type_col = corr_obj.attrs.get("sample_type", "Sample Type")
order_col = corr_obj.attrs.get("inject_order", "Inject Order")
qc_label = corr_obj.attrs.get("sample_dict", {}).get("QC sample", "QC")
actual_label = corr_obj.attrs.get("sample_dict", {}).get("Actual sample", "Sample")
blank_label = corr_obj.attrs.get("sample_dict", {}).get("Blank sample", "Blank")
batch = corr_obj.columns.get_level_values(batch_col).to_numpy(dtype=str)
sample_type = corr_obj.columns.get_level_values(sample_type_col).to_numpy(dtype=str)
order = corr_obj.columns.get_level_values(order_col).to_numpy(dtype=float)
qc_mask = sample_type == qc_label
metanorm_fit_mask = sample_type != blank_label
print("Matrix:", corr_obj.shape, "QC samples:", int(qc_mask.sum()), "MetaNorm fit samples:", int(metanorm_fit_mask.sum()), "predicted/corrected blanks:", int((~metanorm_fit_mask).sum()), "batches:", len(np.unique(batch)))

In [ ]:
# Run every native candidate directly through the same dispatcher used by AUTO.
native_candidates = [
    {"method": "QC-RLSC", "label": "QC-RLSC", "params": {"robust": False}},
    {"method": "QC-RLSC", "label": "robust QC-RLSC", "params": {"robust": True}},
    "QC-RFSC", "QC-SVR", "SERRF", "RUV-III", "WaveICA 2.0",
]
native_results = corr_obj._evaluate_correction_candidates(
    methods_to_run=native_candidates,
    batch_array=batch, qc_mask=qc_mask, order_array=order,
    batch_col=batch_col, sample_type_col=sample_type_col, qc_label=qc_label,
)

workflows = {"Raw [Full]": data_obj}
parameter_rows = [{"Workflow": "Raw [Full]", "Engine": "none", "Mode": "Full"}]
for label, result in native_results.items():
    final_full = list(result["stage_dfs"].values())[-1]
    workflows[f"{label} [Full]"] = final_full
    params = dict(result.get("candidate_params", {}))
    parameter_rows.append({
        "Workflow": f"{label} [Full]", "Engine": result.get("method", label),
        "Backend": "numba" if result.get("method") == "QC-RLSC" else "native",
        "Mode": "Full", **{key: params.get(key, corr_obj.attrs.get(key)) for key in ["robust", "loess_span", "loess_degree", "cv_folds", "rf_n_tree", "serrf_n_tree", "ruv_k"]},
    })
    oof_stages = result.get("stage_oof_dfs", {})
    if oof_stages:
        workflows[f"{label} [OOF]"] = list(oof_stages.values())[-1]
        parameter_rows.append({"Workflow": f"{label} [OOF]", "Engine": result.get("method", label), "Backend": "numba" if result.get("method") == "QC-RLSC" else "native", "Mode": "OOF", **{key: params.get(key, corr_obj.attrs.get(key)) for key in ["robust", "loess_span", "loess_degree", "cv_folds", "rf_n_tree", "serrf_n_tree", "ruv_k"]}})

for workflow, matrix in workflows.items():
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", workflow)
    matrix.to_csv(OUTPUT / f"{safe}.csv")
native_parameter_table = pd.DataFrame(parameter_rows)
native_parameter_table.to_csv(OUTPUT / "correction_parameters_native.csv", index=False)
print("Native workflows:", list(workflows))

In [ ]:
# MetaNorm bridge. MetaNorm operates on log2 intensity with keepScale=TRUE.
# Windows R installs need their DLL directories registered before importing rpy2.
import os
_r_paths = [r"D:\R\R-4.5.2\bin\x64", r"D:\R\R-4.5.2\bin", r"D:\rtools45\usr\bin"]
os.environ["R_HOME"] = r"D:\R\R-4.5.2"
os.environ["PATH"] = ";".join(_r_paths + [os.environ.get("PATH", "")])
_r_dll_handles = [os.add_dll_directory(path) for path in _r_paths if os.path.isdir(path)]
def run_metanorm(model: str, qc_only: bool = False, qc_check: bool = False) -> pd.DataFrame:
    try:
        # This R installation's CMD config exits without linker flags when
        # called from PowerShell. Fall back to its known R.dll directory.
        import types
        import rpy2.situation as _rpy2_situation
        try:
            _rpy2_situation.get_r_flags(os.environ["R_HOME"], "--ldflags")
        except Exception:
            def _windows_r_flags(r_home, flags):
                if flags == "--ldflags":
                    return types.SimpleNamespace(l=[], L=[os.path.join(r_home, "bin", "x64")]), []
                return types.SimpleNamespace(l=[], L=[], I=[os.path.join(r_home, "include")]), []
            _rpy2_situation.get_r_flags = _windows_r_flags
        import rpy2.robjects as ro
        from rpy2.robjects import numpy2ri
        from rpy2.robjects.conversion import localconverter
    except ImportError as exc:
        raise RuntimeError("This comparison notebook requires rpy2 in the metaboqc environment.") from exc
    required = ro.r("function(pkg) requireNamespace(pkg, quietly=TRUE)")
    if not bool(required("metanorm")[0]):
        raise RuntimeError("R package metanorm is unavailable. Install it in the metaboqc R library before running this notebook.")
    runner = ro.r(r"""
        function(mat, order, batch, sample_type, model, qc_only, qc_check) {
            suppressPackageStartupMessages(library(metanorm))
            batch <- as.factor(as.character(batch))
            # MetaNorm has no Blank-specific path. Clone its worker so blanks
            # remain available for prediction while never entering datfit or
            # the QCcheck fallback/refit models.
            worker <- metanorm::metanormWorker
            worker_body <- paste(deparse(body(worker)), collapse="\n")
            worker_body <- sub("else \\{[[:space:]]*datfit <- dat[[:space:]]*\\}", "else { datfit <- dat[dat$type != 'Blank', ] }", worker_body, perl=TRUE)
            worker_body <- gsub("dat\\[dat\\$type != \"QC\", \\]", "dat[dat$type != 'QC' & dat$type != 'Blank', ]", worker_body, perl=TRUE)
            body(worker) <- parse(text=worker_body)[[1]]
            out <- lapply(seq_len(nrow(mat)), function(i) {
                tryCatch(worker(
                    raw=unname(mat[i, ]), order=as.numeric(order), keepScale=TRUE,
                    QConly=qc_only, QCcheck=qc_check, QCcheckp=0.1, changepoints=FALSE,
                    type=as.character(sample_type), batch=batch, batchwise=TRUE,
                    weights=rep(1, ncol(mat)), model=model, k=min(ncol(mat)*0.9, 10),
                    cv="GCV", plotdir=NULL, plottype="pdf", i=i
                ), error=function(e) rep(NA_real_, ncol(mat)))
            })
            do.call(rbind, out)
        }
    """)
    values = np.log2(corr_obj.where(corr_obj > 0).to_numpy(dtype=float))
    # All samples stay in `dat` for baseline prediction and correction. The
    # R worker above restricts fitting and QCcheck refits to non-Blank rows.
    r_matrix = ro.r["matrix"](ro.FloatVector(values.ravel(order="F")), nrow=values.shape[0], ncol=values.shape[1])
    with localconverter(ro.default_converter + numpy2ri.converter):
        result = runner(r_matrix, ro.FloatVector(order.tolist()), ro.StrVector(batch.tolist()), ro.StrVector(sample_type.tolist()), model, bool(qc_only), bool(qc_check))
        array = np.asarray(result, dtype=float)
    # rLOESS cannot extrapolate beyond a batch's non-Blank fitting range.
    # Preserve only those unsupported Blank observations instead of emitting
    # missing values or allowing blanks back into the fitted baseline.
    blank_cells = np.broadcast_to(~metanorm_fit_mask, array.shape)
    blank_input_missing = blank_cells & ~np.isfinite(values)
    blank_fallback = blank_cells & np.isfinite(values) & ~np.isfinite(array)
    array[blank_fallback] = values[blank_fallback]
    run_metanorm.last_blank_fallback_count = int(blank_fallback.sum())
    run_metanorm.last_blank_input_missing_count = int(blank_input_missing.sum())
    log_df = pd.DataFrame(array, index=corr_obj.index, columns=corr_obj.columns)
    run_metanorm.last_log_df = log_df
    result_df = pd.DataFrame(np.exp2(array), index=corr_obj.index, columns=corr_obj.columns)
    if result_df.isna().all(axis=None):
        raise RuntimeError(f"MetaNorm {model} returned no finite corrected values.")
    return result_df

metanorm_parameters = []
metanorm_log_results = {}
metanorm_failures = []
for qc_only, qc_check in [(False, False), (False, True), (True, False)]:
    mode = "QConly" if qc_only else ("Full_QCcheck" if qc_check else "Full")
    for model in ["rLOESS", "rGAM", "tGAM"]:
        if qc_check and model == "tGAM":
            metanorm_failures.append({"Model": model, "Mode": mode, "Error": "Skipped: package callr-based QCcheck is prohibitively slow for all 347 features on this Windows dataset."})
            continue
        try:
            corrected = run_metanorm(model, qc_only=qc_only, qc_check=qc_check)
        except RuntimeError as exc:
            metanorm_failures.append({"Model": model, "Mode": mode, "Error": str(exc)})
            continue
        workflow = f"MetaNorm {model} [{mode}]"
        metanorm_log_results[workflow] = run_metanorm.last_log_df.copy()
        workflows[workflow] = corrected
        corrected.to_csv(OUTPUT / f"{workflow.replace(' ', '_')}.csv")
        metanorm_parameters.append({"Workflow": workflow, "Engine": "MetaNorm", "Model": model, "Mode": mode, "QConly": qc_only, "QCcheck": qc_check, "Fit sample types": f"{qc_label}, {actual_label}", "Excluded sample types": blank_label, "Fit samples": int(metanorm_fit_mask.sum()), "Predicted/corrected blanks": int((~metanorm_fit_mask).sum()), "Blank input-missing cells": run_metanorm.last_blank_input_missing_count, "Blank no-extrapolation fallbacks": run_metanorm.last_blank_fallback_count, "InputScale": "log2", "RReturnScale": "log2", "PythonInverse": "exp2", "keepScale": True, "batchwise": True, "cv": "GCV", "k": min(metanorm_fit_mask.sum() * 0.9, 10)})
if metanorm_parameters:
    pd.DataFrame(metanorm_parameters).to_csv(OUTPUT / "correction_parameters_metanorm.csv", index=False)
if metanorm_failures:
    pd.DataFrame(metanorm_failures).to_csv(OUTPUT / "metanorm_failures.csv", index=False)
scale_audit = pd.DataFrame([{"Stage": "Input to R", "Scale": "log2(raw intensity)", "Operation": "np.log2; all samples retained for prediction"}, {"Stage": "metanormWorker output", "Scale": "same log2 scale", "Operation": "keepScale=TRUE; Blank excluded from datfit and QCcheck only"}, {"Stage": "Workflow matrix", "Scale": "raw intensity", "Operation": "np.exp2(R output); unsupported Blank extrapolations retain raw values"}])
scale_audit.to_csv(OUTPUT / "metanorm_scale_audit.csv", index=False)
print("All available workflows:", list(workflows))

In [ ]:
from pimqc.processing.correction import MetaboIntCorrector

def qc_rsd(matrix: pd.DataFrame) -> pd.Series:
    qc = matrix.loc[:, qc_mask].astype(float).replace([np.inf, -np.inf], np.nan)
    values = qc.std(axis=1, ddof=1).div(qc.mean(axis=1).replace(0, np.nan))
    return values.replace([np.inf, -np.inf], np.nan)

summary_rows = []
rsd_table = pd.DataFrame({name: qc_rsd(matrix) for name, matrix in workflows.items()})
scale_rows = []
for name, log_matrix in metanorm_log_results.items():
    log_values = qc_rsd(log_matrix).dropna() * 100.0
    raw_values = qc_rsd(workflows[name]).dropna() * 100.0
    scale_rows.append({"Workflow": name, "Median QC-RSD on log2 output (%)": log_values.median(), "Median QC-RSD after exp2 (%)": raw_values.median(), "Finite features log2": int(log_values.size), "Finite features raw": int(raw_values.size)})
pd.DataFrame(scale_rows).to_csv(OUTPUT / "metanorm_scale_sensitivity.csv", index=False)
for name in rsd_table.columns:
    values = rsd_table[name].dropna() * 100.0
    summary_rows.append({
        "Workflow": name, "Features evaluated": int(values.size),
        "Median QC-RSD (%)": values.median(), "IQR QC-RSD (%)": values.quantile(0.75) - values.quantile(0.25),
        "Mean QC-RSD (%)": values.mean(), "P90 QC-RSD (%)": values.quantile(0.90),
        "QC-RSD <15% (%)": (values < 15).mean() * 100, "QC-RSD <30% (%)": (values < 30).mean() * 100,
        "Median change from raw (%)": (1 - values.div(rsd_table["Raw [Full]"].loc[values.index] * 100)).median() * 100,
    })
summary = pd.DataFrame(summary_rows).sort_values("Median QC-RSD (%)")
summary.to_csv(OUTPUT / "correction_qc_rsd_summary.csv", index=False)
rsd_table.to_csv(OUTPUT / "correction_featurewise_qc_rsd.csv")

# Paired feature-level comparisons make the ranking auditable and avoid
# interpreting medians computed on different finite-feature subsets.
paired_rows = []
workflow_names = [name for name in rsd_table.columns if name != "Raw [Full]"]
from scipy.stats import wilcoxon
for left_index, left_name in enumerate(workflow_names):
    for right_name in workflow_names[left_index + 1:]:
        pair = rsd_table[[left_name, right_name]].dropna()
        if pair.empty:
            continue
        left_values = pair[left_name].to_numpy(dtype=float)
        right_values = pair[right_name].to_numpy(dtype=float)
        try:
            p_value = float(wilcoxon(left_values, right_values).pvalue)
        except ValueError:
            p_value = np.nan
        paired_rows.append({
            "Left workflow": left_name, "Right workflow": right_name,
            "Features evaluated": int(len(pair)),
            "Left lower QC-RSD": int((left_values < right_values).sum()),
            "Right lower QC-RSD": int((right_values < left_values).sum()),
            "Ties": int(np.isclose(left_values, right_values, rtol=1e-12, atol=1e-12).sum()),
            "Median left-minus-right (percentage points)": float(np.median((left_values - right_values) * 100.0)),
            "Wilcoxon paired p-value": p_value,
        })
paired_comparison = pd.DataFrame(paired_rows)
paired_comparison.to_csv(OUTPUT / "correction_qc_rsd_paired_comparison.csv", index=False)
coverage = summary[["Workflow", "Features evaluated"]].copy()
coverage["Coverage (%)"] = coverage["Features evaluated"] / len(corr_obj.index) * 100.0
coverage.to_csv(OUTPUT / "correction_qc_rsd_coverage.csv", index=False)
parameters = pd.concat([native_parameter_table, pd.DataFrame(metanorm_parameters)], ignore_index=True, sort=False)
parameters.to_csv(OUTPUT / "correction_parameters_all.csv", index=False)
display(summary)
display(parameters)
display(paired_comparison.head())

In [ ]:
# Assessment-compatible visual checks: QC-RSD distribution and PCA diagnostics.
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7.0, 3.8))
plot_df = rsd_table.melt(var_name="Workflow", value_name="QC-RSD").dropna()
plot_df["QC-RSD"] *= 100
order_names = summary["Workflow"].tolist()
sns.boxplot(data=plot_df, x="QC-RSD", y="Workflow", order=order_names, fliersize=0.5, linewidth=0.5, ax=ax)
ax.axvline(15, color="tab:green", linestyle="--", linewidth=0.8)
ax.axvline(30, color="tab:orange", linestyle=":", linewidth=0.8)
ax.set_xlabel("Feature-wise QC RSD (%)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(OUTPUT / "qc_rsd_workflow_comparison.png", dpi=300, bbox_inches="tight")
plt.close(fig)

from pimqc.processing.assessment import MetaboIntAssessor, MetaboVisualizerAssessor
pca_rows = []
for name, matrix in workflows.items():
    try:
        assessor = MetaboIntAssessor(matrix, pipeline_params=PIPELINE_PARAMS)
        assessor.attrs.update(data_obj.attrs)
        diagnostics = assessor.pca_res.get("diagnostics", {})
        pca_rows.append({"Workflow": name, **{key: diagnostics.get(key) for key in ["relative_dispersion", "batch_silhouette", "centrality_shift"]}})
    except Exception as exc:
        pca_rows.append({"Workflow": name, "Assessment error": str(exc)})
pca_summary = pd.DataFrame(pca_rows)
pca_summary.to_csv(OUTPUT / "assessment_structure_metrics.csv", index=False)
display(pca_summary)
# Reuse the assessment PCA renderer for a representative visual per workflow.
actual_label = data_obj.attrs.get("sample_dict", {}).get("Actual sample", "Sample")
for name, matrix in workflows.items():
    try:
        assessor = MetaboIntAssessor(matrix, pipeline_params=PIPELINE_PARAMS)
        assessor.attrs.update(data_obj.attrs)
        pca_res = assessor.pca_res
        visualizer = MetaboVisualizerAssessor(assessor)
        fig = visualizer.plot_pca_scatter(
            pca_df=pca_res["pca_scatter"], pca_var=pca_res["pca_variance"],
            pca_diagnostics=pca_res["diagnostics"], sample_type=sample_type_col,
            batch=batch_col, qc_label=qc_label, actual_label=actual_label,
        )
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
        fig.savefig(OUTPUT / f"assessment_pca_{safe}.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
    except Exception as exc:
        print(f"PCA plot failed for {name}: {exc}")
print("Results written to", OUTPUT)